Topic: PySpark Data Cleaning - Read CSV and Handle Null Values

Objective:
This script demonstrates a basic PySpark ETL workflow:
1. Read raw customer CSV data from a Databricks Volume
2. Inspect schema and record count
3. Remove duplicate records
4. Trim string columns
5. Filter required null or blank values
6. Write cleaned output as Parquet

Why this matters:
Raw data in real-world data engineering pipelines is rarely clean. Data engineers must validate,
clean, standardize, and store data in optimized formats before it reaches analytics or reporting layers.

Architecture / Flow
Raw CSV in Databricks Volume
        ↓
Read using Spark DataFrame
        ↓
Inspect schema and count
        ↓
Remove duplicates
        ↓
Trim string columns
        ↓
Filter null and blank values
        ↓
Write clean output as Parquet

In [0]:

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, trim


spark = SparkSession.builder.appName("daily-practice").getOrCreate()


INPUT_PATH = "/Volumes/workspace/data/daily_practice/data/customers.csv"
OUTPUT_PATH = "/Volumes/workspace/data/daily_practice/output/customers_cleaned"


raw_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(INPUT_PATH)
)


print("Raw Data")
display(raw_df)

print("Raw Schema")
raw_df.printSchema()

print(f"Raw Record Count: {raw_df.count()}")




Raw Data


customer_id,customer_name,email,city
1,Avishek Bhandari,avishek@example.com,Detroit
2,John Smith,john@example.com,Chicago
3,null,missingname@example.com,Boston
4,Sara Khan,,New York
5,Avishek Bhandari,avishek@example.com,Detroit
6,Maria Lopez,maria@example.com,Dallas


Raw Schema
root
 |-- customer_id: integer (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- city: string (nullable = true)

Raw Record Count: 6


In [0]:
cleaned_df = (
    raw_df
    .dropDuplicates()
    .withColumn("customer_name", trim(col("customer_name")))
    .withColumn("email", trim(col("email")))
    .withColumn("city", trim(col("city")))
    .filter(col("customer_id").isNotNull())
    .filter((col("customer_name").isNotNull()) & (col("customer_name") != ""))
    .filter((col("email").isNotNull()) & (col("email") != ""))
)


print("Cleaned Data")
display(cleaned_df)

print(f"Cleaned Record Count: {cleaned_df.count()}")

Cleaned Data


customer_id,customer_name,email,city
1,Avishek Bhandari,avishek@example.com,Detroit
2,John Smith,john@example.com,Chicago
5,Avishek Bhandari,avishek@example.com,Detroit
6,Maria Lopez,maria@example.com,Dallas


Cleaned Record Count: 4


In [0]:

(cleaned_df
 .write.mode("overwrite")
 .parquet(OUTPUT_PATH)
)

print(f"Cleaned data written successfully to: {OUTPUT_PATH}")

Cleaned data written successfully to: /Volumes/workspace/data/daily_practice/output/customers_cleaned
